# Instituto Superior de Engenharia do Porto (ISEP)
## Licenciatura em Engenharia Física Aplicada (LEFA)
### Mecânica e Relatividade (MECREL) — 2026/2027
---
# Notebook TP06: Trabalho, Energia Mecânica, Forças Conservativas e Poços de Potencial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristovaoDias/LEFA-MECREL/blob/main/Notebooks_Colab/TP06_Trabalho_Energia_Potenciais_Colab.ipynb)

**Docente:** Cristóvão de Sousa Dias (`css@isep.ipp.pt`)  
**Objetivos Pedagógicos:**
1. **Trabalho de Forças Variáveis:** Calcular numericamente o trabalho mecânico via quadratura de Riemann/Trapézios e validar o Teorema do Trabalho-Energia ($W_{\text{total}} = \Delta K$).
2. **Cartografia de Poços de Potencial:** Mapear curvas de energia potencial $U(x)$, obter forças conservativas $F(x) = -dU/dx$ e identificar regiões permitidas, proibidas e pontos de retorno ($U(x) = E$).
3. **Física Molecular e Nanotecnologia:** Simular o Potencial de Lennard-Jones para o dímero de Árgon ($Ar_2$), explorando o poço molecular, a constante elástica efetiva $k_{\text{eff}}$ e a aproximação de pequenas oscilações harmónicas.
4. **Auditoria Física a Modelos GenAI (Problema E.2):** Diagnosticar a violação sistemática do princípio de conservação de energia em scripts gerados por IA baseados no método de Euler explícito, e implementar um integrador simplético (Verlet de Velocidades) com tolerância $|\Delta E/E_0| \le 0.05\%$.

## 1. Setup & Configuração Científica

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid, cumulative_trapezoid

# Configuração do layout gráfico institucional do ISEP
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.35
print("✓ Ambiente científico configurado com sucesso!")

## 2. Módulo 1: Trabalho de Forças Variáveis e Teorema do Trabalho-Energia

Consideremos o sistema de travagem de emergência de um caça militar num porta-aviões (Problema B.2 da Ficha TP06).
O cabo de retenção elástico-hidráulico exerce uma força de desaceleração não linear dada por:
$$F_x(x) = -k x - c x^2$$
onde $k = 4.0 \times 10^4\text{ N/m}$, $c = 1.2 \times 10^3\text{ N/m}^2$, e a massa do caça é $M = 16\,000\text{ kg}$ com velocidade inicial $v_0 = 65.0\text{ m/s}$ ($234\text{ km/h}$).

O trabalho realizado pelo cabo desde $x = 0$ até uma posição arbitrária $x$ é dado por:
$$W(x) = \int_0^x F_x(x') dx' = -\frac{1}{2} k x^2 - \frac{1}{3} c x^3$$
Pelo Teorema do Trabalho-Energia, a velocidade do caça ao longo da desaceleração é:
$$K(x) - K_0 = W(x) \implies v(x) = \sqrt{v_0^2 + \frac{2}{M} W(x)}$$

In [ ]:
# Parâmetros do sistema de travagem
M = 16000.0       # kg
v0 = 65.0         # m/s
k = 4.0e4         # N/m
c = 1.2e3         # N/m^2
K0 = 0.5 * M * v0**2

# Determinação da distância de paragem d: 0.5*k*d^2 + (1/3)*c*d^3 = K0
# c/3 * d^3 + k/2 * d^2 - K0 = 0
poly_coeffs = [c / 3.0, k / 2.0, 0.0, -K0]
roots = np.roots(poly_coeffs)
real_roots = [r.real for r in roots if np.isreal(r) and r.real > 0]
d_stop = real_roots[0]
print(f"✓ Energia Cinética Inicial: K0 = {K0/1e6:.2f} MJ")
print(f"✓ Distância de Paragem Calculada: d = {d_stop:.2f} m")

# Vetor de posições até à paragem
x = np.linspace(0, d_stop, 300)
F_res = -(k * x + c * x**2)
W_analitico = -0.5 * k * x**2 - (c / 3.0) * x**3
W_numerico = cumulative_trapezoid(F_res, x, initial=0)

# Velocidade em função da distância percorrida
v_x = np.sqrt(np.maximum(0, v0**2 + (2.0 / M) * W_numerico))

# Visualização Gráfica Dupla
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Força de Retenção e Trabalho (Área)
ax1.plot(x, -F_res / 1e3, color='#EF4444', lw=2.5, label='Força de Travagem $|F_x(x)|$')
ax1.fill_between(x, 0, -F_res / 1e3, color='#EF4444', alpha=0.15, label='Trabalho Dissipado $|W|$')
ax1.axvline(d_stop, color='#0A2540', ls='--', lw=1.5, label=f'Paragem: $d = {d_stop:.1f}\text{{ m}}$')
ax1.set_title('Perfil da Força de Retenção e Trabalho', fontweight='bold', color='#0A2540')
ax1.set_xlabel('Deslocamento $x$ (m)', fontweight='bold')
ax1.set_ylabel('Força $|F_x|$ (kN)', fontweight='bold')
ax1.legend()

# Gráfico 2: Perfil de Velocidade v(x)
ax2.plot(x, v_x * 3.6, color='#0077B6', lw=2.5, label='Velocidade $v(x)$ (km/h)')
ax2.axhline(0, color='gray', lw=1)
ax2.axvline(d_stop, color='#0A2540', ls='--', lw=1.5, label=f'Imobilização total')
ax2.set_title('Perfil de Desaceleração: $v(x)$', fontweight='bold', color='#0A2540')
ax2.set_xlabel('Deslocamento $x$ (m)', fontweight='bold')
ax2.set_ylabel('Velocidade (km/h)', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Módulo 2: Cartografia de Curvas de Energia Potencial $U(x)$ e Retorno

Dada uma energia mecânica total $E = K + U(x)$, como a energia cinética é estritamente não negativa ($K \ge 0$):
$$K = E - U(x) \ge 0 \implies E \ge U(x)$$
- **Região Classicamente Permitida:** Onde $U(x) \le E$. A velocidade é $v(x) = \pm \sqrt{\frac{2(E - U(x))}{m}}$.
- **Pontos de Retorno ($x_{\text{ret}}$):** Onde $U(x_{\text{ret}}) = E \implies K = 0 \implies v = 0$. A partícula inverte o sentido do movimento.
- **Região Classicamente Proibida:** Onde $U(x) > E$. A partícula não pode penetrar sem violar a física clássica.
- **Força e Equilíbrio:** $F(x) = -\frac{dU}{dx}$.
  - Mínimo de $U(x)$ ($\frac{d^2U}{dx^2} > 0$): **Equilíbrio Estável** (força restauradora).
  - Máximo de $U(x)$ ($\frac{d^2U}{dx^2} < 0$): **Equilíbrio Instável** (força repulsiva).

In [ ]:
# Definição de um poço duplo cúbico/quártico assimétrico
def U_potencial(x):
    return x**4 - 3.0 * x**2 - 0.5 * x

def F_forca(x):
    # F(x) = -dU/dx = -(4*x^3 - 6*x - 0.5)
    return -(4.0 * x**3 - 6.0 * x - 0.5)

x_vals = np.linspace(-2.2, 2.2, 500)
U_vals = U_potencial(x_vals)
F_vals = F_forca(x_vals)

# Energia mecânica de referência para a partícula
E_total = 1.0  # Joules

plt.figure(figsize=(11, 5.5))
plt.plot(x_vals, U_vals, color='#0077B6', lw=2.5, label='Potencial $U(x) = x^4 - 3x^2 - 0.5x$')
plt.axhline(E_total, color='#EF4444', lw=2, ls='--', label=f'Energia Total $E = {E_total:.1f}\text{{ J}}$')

# Sombreamento das regiões proibidas
plt.fill_between(x_vals, U_vals, 10, where=(U_vals > E_total), color='gray', alpha=0.25, label='Região Proibida ($U(x) > E$)')
plt.fill_between(x_vals, U_vals, E_total, where=(U_vals <= E_total), color='#10B981', alpha=0.15, label='Região Permitida ($K \\geq 0$)')

# Pontos de equilíbrio: raízes de 4*x^3 - 6*x - 0.5 = 0
eq_roots = np.roots([4.0, 0.0, -6.0, -0.5])
for r in eq_roots:
    if np.isreal(r):
        xr = r.real
        Ur = U_potencial(xr)
        d2U = 12.0 * xr**2 - 6.0
        if d2U > 0:
            plt.plot(xr, Ur, 'o', color='#10B981', ms=9, label=f'Eq. Estável ($x = {xr:.2f}\\text{{ m}}$)')
        else:
            plt.plot(xr, Ur, 's', color='#EF4444', ms=9, label=f'Eq. Instável ($x = {xr:.2f}\\text{{ m}}$)')

plt.ylim(-4, 6)
plt.xlim(-2.2, 2.2)
plt.title('Cartografia de Poço Duplo de Potencial e Estabilidade', fontweight='bold', color='#0A2540')
plt.xlabel('Posição $x$ (m)', fontweight='bold')
plt.ylabel('Energia (J)', fontweight='bold')
plt.legend(loc='lower center', ncol=2, framealpha=0.9)
plt.tight_layout()
plt.show()

## 4. Módulo 3: Potencial Molecular de Lennard-Jones (Árgon $Ar_2$)

O potencial de Lennard-Jones modela a ligação atómica neutra entre dois átomos (ex: dímero de gás nobre $Ar_2$):
$$U(r) = 4\epsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6 \right]$$
- Parâmetros empíricos para o Árgon: $\epsilon = 1.65 \times 10^{-21}\text{ J} = 10.3\text{ meV}$, $\sigma = 0.340\text{ nm} = 3.40\text{ Å}$.
- A força interatómica é:
$$F(r) = -\frac{dU}{dr} = \frac{24\epsilon}{\sigma} \left[ 2\left(\frac{\sigma}{r}\right)^{13} - \left(\frac{\sigma}{r}\right)^7 \right]$$
- Posição de equilíbrio estável: $F(r_0) = 0 \implies r_0 = 2^{1/6}\sigma \approx 1.1225 \,\sigma$.
- No fundo do poço, a energia é mínima: $U(r_0) = -\epsilon$ (energia de dissociação molecular).
- Aproximação de pequenas oscilações: expandindo em Taylor em torno de $r_0$:
$$U(r) \approx -\epsilon + \frac{1}{2} k_{\text{eff}} (r - r_0)^2, \qquad k_{\text{eff}} = \left.\frac{d^2U}{dr^2}\right|_{r_0} = \frac{72\epsilon}{2^{1/3}\sigma^2}$$

In [ ]:
# Constantes do Árgon
sigma = 0.340e-9         # m (0.34 nm)
epsilon_J = 1.65e-21     # J
epsilon_meV = 10.3       # meV
m_Ar = 6.63e-26          # kg (massa do átomo de Árgon)
mu_red = m_Ar / 2.0      # kg (massa reduzida do dímero Ar2)

# Raio de equilíbrio e rigidez efetiva
r0 = 2.0**(1.0/6.0) * sigma
k_eff = (72.0 * epsilon_J) / (2.0**(1.0/3.0) * sigma**2)
f0 = (1.0 / (2.0 * np.pi)) * np.sqrt(k_eff / mu_red)

print(f"✓ Distância de Equilíbrio: r0 = {r0*1e9:.4f} nm ({r0*1e10:.2f} Angstroms)")
print(f"✓ Poço de Potencial: U(r0) = -epsilon = {-epsilon_meV:.1f} meV")
print(f"✓ Rigidez Efetiva do Poço: k_eff = {k_eff:.3f} N/m")
print(f"✓ Frequência Própria de Vibração: f0 = {f0/1e12:.2f} THz")

# Malha radial de cálculo
r = np.linspace(0.31e-9, 0.70e-9, 500)
U_LJ = 4.0 * epsilon_meV * ((sigma / r)**12 - (sigma / r)**6)
U_harm = -epsilon_meV + 0.5 * (k_eff / 1.602e-22) * (r - r0)**2  # convertido para meV

plt.figure(figsize=(10, 5.5))
plt.plot(r * 1e9, U_LJ, color='#0077B6', lw=2.5, label='Potencial de Lennard-Jones Exato $U(r)$')
plt.plot(r * 1e9, U_harm, '--', color='#EF4444', lw=2, label='Aproximação Harmónica Parabólica ($k_{\\text{eff}}$)')
plt.axvline(r0 * 1e9, color='#10B981', ls=':', lw=1.8, label=f'Equilíbrio $r_0 = {r0*1e9:.3f}\text{{ nm}}$')
plt.axhline(-epsilon_meV, color='#F59E0B', ls=':', lw=1.5, label=f'Profundidade $\\epsilon = -{epsilon_meV:.1f}\text{{ meV}}$')

plt.ylim(-15, 20)
plt.xlim(0.31, 0.65)
plt.title('Potencial Molecular de Lennard-Jones ($Ar_2$): Exato vs. Harmónico', fontweight='bold', color='#0A2540')
plt.xlabel('Distância Interatómica $r$ (nm)', fontweight='bold')
plt.ylabel('Energia Potencial $U(r)$ (meV)', fontweight='bold')
plt.legend(framealpha=0.9)
plt.show()

## 5. Módulo 4: Auditoria Física a Modelos GenAI (Problema E.2)

> **Contexto da Falha Algorítmica:**
> A maioria dos Modelos de Linguagem (ChatGPT, Claude, Gemini) recorre por defeito ao método de **Euler Explícito** de 1.ª ordem para resolver equações diferenciais da mecânica:
> $$x_{n+1} = x_n + v_n \Delta t, \qquad v_{n+1} = v_n + a(x_n) \Delta t$$
> Este algoritmo **não preserva o volume no espaço de fase (não é simplético)**. A cada passo temporal, ele introduz energia espúria de ordem $\mathcal{O}(\Delta t)$, fazendo com que a energia mecânica aparente cresça artificialmente ($E_{n+1} > E_n$). Como consequência física, o modelo de IA pode simular partículas a "saltar" barreiras de potencial intransponíveis!

### 5.1 Demonstração do Código Ingénuo Gerado por LLM (Euler Explícito)
Executemos o código padrão gerado por um LLM sem supervisão física com $\Delta t = 0.01\text{ s}$ e $t = 12.0\text{ s}$:

In [ ]:
# Parâmetros do Desafio E.2
m = 1.0
dt = 0.01
t_max = 12.0
time = np.arange(0, t_max, dt)
N = len(time)

# Vetores do método de Euler Explícito (Típico de LLM genérico)
x_euler = np.zeros(N)
v_euler = np.zeros(N)
E_euler = np.zeros(N)

# Condições iniciais: parte do repouso em x0 = 1.8 m
x_euler[0] = 1.8
v_euler[0] = 0.0
E_0 = U_potencial(x_euler[0]) + 0.5 * m * v_euler[0]**2

for n in range(N - 1):
    # F_forca(x) = -(4*x^3 - 6*x - 0.5)
    a_n = F_forca(x_euler[n]) / m
    x_euler[n+1] = x_euler[n] + v_euler[n] * dt
    v_euler[n+1] = v_euler[n] + a_n * dt
    E_euler[n] = U_potencial(x_euler[n]) + 0.5 * m * v_euler[n]**2

E_euler[-1] = U_potencial(x_euler[-1]) + 0.5 * m * v_euler[-1]**2
drift_euler = np.abs(E_euler[-1] - E_0) / np.abs(E_0) * 100.0

print(f"⚠ Energia Inicial E(0): {E_0:.4f} J")
print(f"⚠ Energia Final E(t=12): {E_euler[-1]:.4f} J")
print(f"❌ ERRO CRÍTICO: Deriva Energética no Euler Explícito = +{drift_euler:.1f}%")

# Gráfico comparativo do desastre de conservação de energia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(time, x_euler, color='#EF4444', lw=2, label='Trajetória $x(t)$ (Euler Explícito)')
ax1.set_title('Trajetória Instável: Salto Artificial de Barreira', fontweight='bold', color='#0A2540')
ax1.set_xlabel('Tempo $t$ (s)', fontweight='bold')
ax1.set_ylabel('Posição $x$ (m)', fontweight='bold')
ax1.legend()

ax2.plot(time, E_euler, color='#EF4444', lw=2, label='Energia $E(t)$ (Euler)')
ax2.axhline(E_0, color='#10B981', ls='--', lw=2, label=f'Energia Física Real $E_0 = {E_0:.2f}\text{{ J}}$')
ax2.set_title('Violação da Conservação de Energia (Drift Artificial)', fontweight='bold', color='#0A2540')
ax2.set_xlabel('Tempo $t$ (s)', fontweight='bold')
ax2.set_ylabel('Energia Mecânica $E(t)$ (J)', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

---
### 5.2 Desafio dos Estudantes: Engenharia de Prompts e Auditoria Simplética

> **Instruções para a Turma:**
> 1. **Redige de raiz um Prompt de Engenharia Física** exigindo explicitamente ao teu LLM um integrador simplético conservativo de energia (ex: **Verlet de Velocidades** ou **Euler-Cromer**):
>    - Algoritmo de Verlet de Velocidades:
>      $$x_{n+1} = x_n + v_n \Delta t + \frac{1}{2} a(x_n) \Delta t^2$$
>      $$v_{n+1} = v_n + \frac{1}{2} [a(x_n) + a(x_{n+1})] \Delta t$$
> 2. Submete o teu prompt ao ChatGPT / Claude / Gemini e cola o texto do prompt na **Célula E.2A**.
> 3. Executa o código corrigido na **Célula E.2B**.
> 4. Preenche a tabela de auditoria quantitativa na **Célula E.2C**.

#### Célula E.2A: O Teu Prompt de Engenharia Concebido de Raiz

> *(Copia e cola aqui o texto exato do prompt avançado que submeteste ao modelo)*

#### Célula E.2B: Código Auditado com Integrador Simplético (Velocity Verlet / Euler-Cromer)

In [ ]:
# [ESPAÇO RESERVADO PARA O ESTUDANTE]
# Cole aqui o código gerado pelo LLM contendo o algoritmo simplético (Verlet ou Euler-Cromer).
# Execute e verifique se a energia mecânica se mantém rigorosamente constante ao longo dos 15 segundos!


#### Célula E.2C: Relatório de Auditoria e Diagnóstico Físico

*(Preenche os campos com a tua análise crítica:)*

- **Energia Mecânica Inicial $E(0)$:** `_____ J`
- **Deriva Energética Relativa $|\Delta E / E_0|\%$ no Método de Euler Explícito:** `_____ %`
- **Deriva Energética Relativa no Algoritmo de Verlet:** `_____ %`
- **Comportamento Qualitativo da Partícula:** *(Com o método de Euler, a partícula saltou indevidamente a barreira de potencial? No Verlet, permaneceu corretamente aprisionada no poço da direita?)*
- **Pontos de Retorno Teóricos:** *(Quais são as soluções da equação $U(x) = E_0$ para o poço da direita? Compare com a amplitude máxima observada na oscilação)*: `x_retorno = _____ m`